# April run — smoke test

Exercises **every stage of the April pipeline end-to-end at tiny scope**
(1 day, all three symbols, full predictor/class spec, 5 training epochs) so a
broken step fails here in minutes instead of hours into the real run.
Checks per symbol:

1. data discovery + per-symbol config construction (filter ON) — NVDA/INTC
   from `data/NVDA_INTC`, IBM from its own `data/IBM`
2. distributions stage → **10 `SEQ_DISTR_*` + 50 `CLS_DISTR_*_{cls}`** files
3. ensemble stage (v2) → **20 `ENS_TD_*`** files (3 bivariate + 1 joint
   channel, 5 lengths × 4 classes)
4. training harness → 2 result files + 4 charts for one model

Everything runs in `outputs/april_smoke/` — the real `outputs/april/` and
`configs/april_*.yaml` are untouched. Expected runtime: ~10-20 min on the
compute box (dominated by featurizing 1 day per symbol).

This validates **plumbing, not numbers** — numerical correctness is covered
by the byte-equivalence suites (`verify_cls_v2`, `verify_ensemble_reference`,
`verify_ensemble_v2`, `verify_against_baseline`).

In [ ]:
import shutil, sys, time, pathlib
ROOT = pathlib.Path.cwd()
for p in (ROOT, ROOT / "scripts", ROOT / "tests", ROOT / "TrainingDistributions"):
    sys.path.insert(0, str(p))

from run_april import (find_data_dir, april_dates, detect_pattern,
                       DIST_PREDICTORS, CLS_NAMES, PREDICTORS)
from pipeline.config import RunConfig

SYMBOLS = ["NVDA", "INTC", "IBM"]
SMOKE_EPOCHS = 5          # tiny scope, not the real 3000
SMOKE_SEED = 0            # reproducible so a smoke failure is repeatable
SMOKE = ROOT / "outputs" / "april_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)

# each symbol resolves its own directory (NVDA/INTC share NVDA_INTC; IBM is
# separate); the day-glob only runs once per distinct directory
resolved = {s: find_data_dir(s) for s in SYMBOLS}
scope_by_dir = {}
for s, data_dir in resolved.items():
    if data_dir not in scope_by_dir:
        pattern = detect_pattern(data_dir)
        dates = april_dates(data_dir, pattern)[:1]          # 1 day = tiny scope
        scope_by_dir[data_dir] = (pattern, dates)
        print(f"data: {data_dir}  (pattern: {pattern})  smoke day: {dates}")


def smoke_config(symbol):
    """Same shape as run_april.make_config, redirected to the smoke dir."""
    data_dir = resolved[symbol]
    pattern, dates = scope_by_dir[data_dir]
    cfg = RunConfig()
    cfg.data.symbol = symbol
    cfg.data.data_path = str(data_dir)
    cfg.data.dates = list(dates)
    cfg.data.file_pattern = pattern
    cfg.data.instrument_filter = True
    cfg.distributions.predictors = list(DIST_PREDICTORS)
    cfg.distributions.class_names = list(CLS_NAMES)
    cfg.distributions.class_values = [-1, 0, 1]
    cfg.distributions.output_dir = str((SMOKE / symbol).relative_to(ROOT))
    cfg.featurize.cache_dir = str((SMOKE / symbol / "feature_cache").relative_to(ROOT))
    cfg.ensemble.output_dir = str((SMOKE / symbol / "ensemble").relative_to(ROOT))
    cfg.training.model_dir = str((SMOKE / symbol / "models").relative_to(ROOT))
    # smoke scope, set BEFORE saving so the persisted YAML describes the run
    # that actually executes (CLAUDE.md rule 4) rather than the full-scale
    # defaults -- cell 4 must not override these afterwards
    cfg.training.predictor = PREDICTORS[0]
    cfg.training.epochs = SMOKE_EPOCHS
    cfg.training.seed = SMOKE_SEED
    (SMOKE / symbol).mkdir(parents=True, exist_ok=True)
    cfg.save(SMOKE / f"{symbol.lower()}.yaml")
    return cfg

configs = {s: smoke_config(s) for s in SYMBOLS}
print("PASS  configs built (filter ON, 10 predictors, 5 classes)")

In [ ]:
from pipeline.runner import run

for symbol, cfg in configs.items():
    t0 = time.time()
    run(cfg, run_id=f"smoke-dist-{symbol}")
    out = SMOKE / symbol
    exp_seq = len(cfg.distributions.predictors)
    exp_cls = exp_seq * len(cfg.distributions.class_names)
    pred = cfg.distributions.predicted
    n_seq = len(list(out.glob("SEQ_DISTR_*")))
    n_cls = len(list(out.glob("CLS_DISTR_*")))
    v2_named = len(list(out.glob(f"CLS_DISTR_{symbol}__{pred}-*_202504_*")))
    assert n_seq == exp_seq, f"{symbol}: expected {exp_seq} SEQ, got {n_seq}"
    assert n_cls == exp_cls, f"{symbol}: expected {exp_cls} CLS, got {n_cls}"
    assert v2_named == exp_cls, f"{symbol}: CLS files not in v2 naming"
    print(f"PASS  {symbol} distributions: {n_seq} SEQ + {n_cls} CLS "
          f"(v2 names) in {time.time()-t0:.0f}s  [expected counts derived "
          f"from the config]")

In [ ]:
from pipeline.ensemble import run_ensemble

for symbol, cfg in configs.items():
    t0 = time.time()
    outputs = run_ensemble(cfg, run_id=f"smoke-ens-{symbol}")
    exp_ens = len(cfg.ensemble.seq_lengths) * len(cfg.ensemble.class_names)
    n_ch = len(cfg.ensemble.predictors)
    # file suffix follows the vendored reference the config selects
    suffix = "ALL" if cfg.ensemble.reference == "v2" else f"ALL_{n_ch}"
    n_ens = len(list((SMOKE / symbol / "ensemble").glob(f"ENS_TD_*_{suffix}")))
    assert n_ens == exp_ens, f"{symbol}: expected {exp_ens} ENS_TD, got {n_ens}"
    print(f"PASS  {symbol} ensemble ({cfg.ensemble.reference}): {n_ens} ENS_TD "
          f"files ({suffix} names) in {time.time()-t0:.0f}s  "
          f"[expected counts derived from the config]")

In [ ]:
from pipeline.models import train_model

for symbol in SYMBOLS:
    # every training parameter already comes from the saved config; nothing
    # is overridden here, so the printed/saved YAML is what ran
    cfg = configs[symbol]
    r = train_model(cfg, run_id=f"april-smoke-train-{symbol}")
    assert pathlib.Path(r["model_file"]).exists()
    assert pathlib.Path(r["weights_file"]).exists()
    assert len(r["plots"]) == 4, f"expected 4 charts, got {len(r['plots'])}"
    print(f"PASS  {symbol} training: 2 result files + 4 charts "
          f"({r['n_sequences']} examples, {r['train_seconds']:.0f}s)")

print("\nTrainer OK (direct call). Fan-out is checked in the next cell.")

### 5 — stage-3 fan-out (the way `april_run.ipynb` actually trains)
The cell above calls `train_model` directly. The real run instead goes through
`plan_training` -> `run_training_jobs`, which reads the schedule from the config
and dispatches one training per device in a spawned worker pool. That path is
what full scale uses, so it is exercised here at smoke scope: same 5 epochs,
re-training the same models over the top.

Checks the parts only the fan-out has: schedule read from
`training.gpus`/`training.max_parallel`, one job per (symbol, predictor),
device assignment, results arriving in completion order, and each job's charts
surviving the spawned process.

In [ ]:
from run_april import plan_training, run_training_jobs

smoke_cfgs = {s: SMOKE / f"{s.lower()}.yaml" for s in SYMBOLS}
jobs, n_par = plan_training(smoke_cfgs)        # schedule from the CONFIG
print(f"{len(jobs)} jobs | {n_par} at a time")
for j in jobs:
    print(f"    {j['symbol']:6s} x {j['label']:16s} -> {j['device']}")
assert len(jobs) == len(SYMBOLS), jobs         # 1 train predictor per symbol
assert all(isinstance(j["label"], str) for j in jobs)
assert len({j["run_id"] for j in jobs}) == len(jobs), "run_ids must be unique"

seen = []
for r in run_training_jobs(jobs, n_par):       # spawned workers
    seen.append((r["symbol"], r["predictor"]))
    assert pathlib.Path(r["model_file"]).exists(), r
    assert pathlib.Path(r["weights_file"]).exists(), r
    assert not r.get("chart_error"), r["chart_error"]
    assert len(r["plots"]) == 4, f"{r['symbol']}: {len(r['plots'])} charts"
    print(f"PASS  {r['symbol']} x {r['predictor']} on {r['device']} "
          f"({r['train_seconds']:.0f}s)")

assert len(seen) == len(jobs), f"{len(seen)} results for {len(jobs)} jobs"
assert len(set(seen)) == len(seen), f"duplicate results: {seen}"
print("\nFAN-OUT OK — plan_training + run_training_jobs work end to end.")
print("Safe to run april_run.ipynb at full scope.")